In [1]:
import os

SRC_DIR = "/kaggle/input/datasets/swetha344/pyg-210-cu128-wheels"
DST_DIR = "/kaggle/working/fixed_wheels"

os.makedirs(DST_DIR, exist_ok=True)

rename_map = {
    "torch_scatter-2.1.2pt210cu128-cp312-cp312-linux_x86_64.whl": "torch_scatter-2.1.2+pt210cu128-cp312-cp312-linux_x86_64.whl",
    "torch_cluster-1.6.3pt210cu128-cp312-cp312-linux_x86_64.whl": "torch_cluster-1.6.3+pt210cu128-cp312-cp312-linux_x86_64.whl",
    "pyg_lib-0.4.0pt210cu128-cp312-cp312-linux_x86_64.whl": "pyg_lib-0.4.0+pt210cu128-cp312-cp312-linux_x86_64.whl",
    "torch_geometric-2.7.0-py3-none-any.whl": "torch_geometric-2.7.0-py3-none-any.whl",
    "ogb-1.3.6-py3-none-any.whl": "ogb-1.3.6-py3-none-any.whl"
}

for old_name, new_name in rename_map.items():
    src = os.path.join(SRC_DIR, old_name)
    dst = os.path.join(DST_DIR, new_name)
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)

!pip install --no-index --no-deps --force-reinstall /kaggle/working/fixed_wheels/*.whl

Processing ./fixed_wheels/ogb-1.3.6-py3-none-any.whl
Processing ./fixed_wheels/torch_cluster-1.6.3+pt210cu128-cp312-cp312-linux_x86_64.whl
Processing ./fixed_wheels/torch_geometric-2.7.0-py3-none-any.whl
Processing ./fixed_wheels/torch_scatter-2.1.2+pt210cu128-cp312-cp312-linux_x86_64.whl


In [2]:
import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

In [3]:
import torch
import torch_geometric

# We must use the exact internal paths PyG uses
from torch_geometric.data.data import DataEdgeAttr
from torch_geometric.data.storage import NodeStorage, EdgeStorage, GlobalStorage

# Register them globally with PyTorch's security layer
torch.serialization.add_safe_globals([
    DataEdgeAttr, 
    NodeStorage, 
    EdgeStorage, 
    GlobalStorage,
    torch_geometric.data.Data  # Sometimes needed for the base class
])

# NOW import and run OGB
from ogb.nodeproppred import PygNodePropPredDataset
import torch_geometric.transforms as T

with torch.serialization.safe_globals([DataEdgeAttr, NodeStorage, EdgeStorage]):
    dataset = PygNodePropPredDataset(name='ogbn-arxiv', root='dataset/', transform=T.ToUndirected())

Downloaded 0.08 GB: 100%|██████████| 81/81 [00:02<00:00, 30.48it/s]
Processing...


Extracting dataset/arxiv.zip
Loading necessary files...
This might take a while.
Processing graphs...


100%|██████████| 1/1 [00:00<00:00, 9731.56it/s]


Converting graphs into PyG objects...


100%|██████████| 1/1 [00:00<00:00, 261.80it/s]

Saving...



Done!
/usr/local/lib/python3.12/dist-packages/ogb/nodeproppred/dataset_pyg.py:69: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  self.data, self.slices = torch.load(self.processed_paths[0])


In [4]:
# 1. Basic Dataset Info
print(f"Number of graphs: {len(dataset)}")
print(f"Number of classes: {dataset.num_classes}")
print(f"Number of node features: {dataset.num_node_features}")

# 2. Get the actual Data object
data = dataset[0]

# 3. View Graph Structure
print("\n--- Graph Structure ---")
print(f"Number of nodes: {data.num_nodes}")
print(f"Number of edges: {data.num_edges}")
print(f"Is undirected: {data.is_undirected()}")
print(f"Has isolated nodes: {data.has_isolated_nodes()}")
print(f"Has self-loops: {data.has_self_loops()}")

# 4. Print one Node's Features
# Nodes are indexed. Let's look at node 0.
print("\n--- Node 0 Inspection ---")
print(f"Features (first 5 values): {data.x[0][:5]}")
print(f"Label: {data.y[0].item()}")

# 5. Inspect the Edge Index
# This is a [2, num_edges] tensor representing (source, target)
print("\n--- Edge Index ---")
print(data.edge_index)

Number of graphs: 1
Number of classes: 40
Number of node features: 128

--- Graph Structure ---
Number of nodes: 169343
Number of edges: 2315598
Is undirected: True
Has isolated nodes: False
Has self-loops: False

--- Node 0 Inspection ---
Features (first 5 values): tensor([-0.0579, -0.0525, -0.0726, -0.0266,  0.1304])
Label: 4

--- Edge Index ---
tensor([[     0,      0,      0,  ..., 169341, 169342, 169342],
        [   411,    640,   1162,  ..., 163274,  27824, 158981]])


In [5]:
split_idx = dataset.get_idx_split()
train_idx, valid_idx, test_idx = split_idx["train"], split_idx["valid"], split_idx["test"]

print(f"\nTraining nodes: {len(train_idx)}")
print(f"Validation nodes: {len(valid_idx)}")
print(f"Test nodes: {len(test_idx)}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = data.to(device)


Training nodes: 90941
Validation nodes: 29799
Test nodes: 48603


In [6]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
import torch.nn as nn

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers, dropout):
        super(GraphSAGE, self).__init__()
        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()
        
        # Input Layer
        self.convs.append(SAGEConv(in_channels, hidden_channels, normalize=True))
        self.bns.append(nn.BatchNorm1d(hidden_channels))
        
        # Hidden Layers
        for _ in range(num_layers - 1):
            self.convs.append(SAGEConv(hidden_channels, hidden_channels, normalize=True))
            self.bns.append(nn.BatchNorm1d(hidden_channels))
            
        # Final Fusion Layer (Jumping Knowledge)
        # We concatenate all layers, so the input is hidden_channels * num_layers
        self.final_lin = nn.Linear(hidden_channels * num_layers, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        layer_outputs = []
        
        for i in range(len(self.convs)):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            layer_outputs.append(x)
            
        # Jumping Knowledge: Concatenate all layer outputs
        x = torch.cat(layer_outputs, dim=-1)
        x = self.final_lin(x)
        
        return x
        
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion = torch.nn.CrossEntropyLoss()

In [7]:
import torch
import numpy as np
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Configuration
num_runs = 10
epochs = 500
eval_steps = 10  # Increased frequency to catch the best peaks
all_test_results = []

global_best_test_acc = 0.0
global_save_path = 'overall_best_graphsage_model.pt'

# Using Label Smoothing to bridge the Val/Test gap
criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.1)

for run in range(num_runs):
    seed = run 
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    
    print(f"\n>>> Starting Run {run+1}/{num_runs} (Seed: {seed})")
    
    # 1. Model init with larger capacity but higher dropout
    model = GraphSAGE(
        in_channels=dataset.num_node_features,
        hidden_channels=256, 
        out_channels=dataset.num_classes,
        num_layers=3,
        dropout=0.5
    ).to(device)

    # 2. Optimizer and Scheduler
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
    
    # mode='max' because we want to maximize validation accuracy
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10, min_lr=1e-5)
    
    best_run_val_acc = 0
    best_run_test_acc = 0

    # 3. Training Loop
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = criterion(out[train_idx], data.y[train_idx].squeeze())
        loss.backward()
        optimizer.step()

        # Check performance more frequently
        if epoch % eval_steps == 0:
            model.eval()
            with torch.no_grad():
                logits = model(data.x, data.edge_index)
                y_pred = logits.argmax(dim=-1, keepdim=True)
                
                val_acc = (y_pred[valid_idx] == data.y[valid_idx]).sum().item() / valid_idx.size(0)
                test_acc = (y_pred[test_idx] == data.y[test_idx]).sum().item() / test_idx.size(0)

                # Step the scheduler based on validation accuracy
                scheduler.step(val_acc)
                
                # Log current learning rate (optional, helpful for debugging)
                current_lr = optimizer.param_groups[0]['lr']

                if epoch % 10 == 0:
                    print(f"Epoch {epoch:03d}: Val: {val_acc:.4f}, Test: {test_acc:.4f}, LR: {current_lr:.6f}")
                
                # Best model tracking
                if val_acc > best_run_val_acc:
                    best_run_val_acc = val_acc
                    best_run_test_acc = test_acc

                    if test_acc > global_best_test_acc:
                        global_best_test_acc = test_acc
                        torch.save(model.state_dict(), global_save_path)

    all_test_results.append(best_run_test_acc)
    print(f"Run {run+1} Finished. Best Test Acc for this run: {best_run_test_acc:.4f}")

# 4. Final Reporting
mean_acc = np.mean(all_test_results)
std_acc = np.std(all_test_results)

print("\n" + "="*30)
print(f"FINAL METRICS OVER {num_runs} RUNS")
print(f"Test Accuracy: {mean_acc:.4f} ± {std_acc:.4f}")
print("="*30)


>>> Starting Run 1/10 (Seed: 0)
Epoch 010: Val: 0.4677, Test: 0.4915, LR: 0.005000
Epoch 020: Val: 0.6035, Test: 0.6221, LR: 0.005000
Epoch 030: Val: 0.6627, Test: 0.6508, LR: 0.005000
Epoch 040: Val: 0.6930, Test: 0.6858, LR: 0.005000
Epoch 050: Val: 0.6806, Test: 0.6524, LR: 0.005000
Epoch 060: Val: 0.7037, Test: 0.6834, LR: 0.005000
Epoch 070: Val: 0.7152, Test: 0.6992, LR: 0.005000
Epoch 080: Val: 0.7141, Test: 0.7001, LR: 0.005000
Epoch 090: Val: 0.7219, Test: 0.7075, LR: 0.005000
Epoch 100: Val: 0.7193, Test: 0.7027, LR: 0.005000
Epoch 110: Val: 0.7210, Test: 0.7101, LR: 0.005000
Epoch 120: Val: 0.7146, Test: 0.6994, LR: 0.005000
Epoch 130: Val: 0.7148, Test: 0.7105, LR: 0.005000
Epoch 140: Val: 0.7234, Test: 0.7202, LR: 0.005000
Epoch 150: Val: 0.7107, Test: 0.6918, LR: 0.005000
Epoch 160: Val: 0.7265, Test: 0.7137, LR: 0.005000
Epoch 170: Val: 0.7235, Test: 0.7173, LR: 0.005000
Epoch 180: Val: 0.7217, Test: 0.7068, LR: 0.005000
Epoch 190: Val: 0.7205, Test: 0.7033, LR: 0.00500